In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e1/sample_submission.csv
/kaggle/input/playground-series-s6e1/train.csv
/kaggle/input/playground-series-s6e1/test.csv
/kaggle/input/s6e1-xgboost-1/saved_models/xgboost_model_20260122_152057.joblib
/kaggle/input/s6e1-xgboost-1/saved_models/xgboost_model_20260122_152057.json
/kaggle/input/s6e1-xgboost-1/saved_models/metrics_20260122_152057.json
/kaggle/input/s6e1-xgboost-1/saved_models/xgboost_model_20260122_152057.pkl


In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, PolynomialFeatures
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pickle
import joblib
import json
from datetime import datetime
import pandas as pd

In [3]:
# Load the model
#model = joblib.load('/kaggle/input/s6e1-xgboost-1/saved_models/xgboost_model_20260122_152057.joblib')

In [4]:
#model

In [5]:
def engineer_features(df):
    # Create new features
    df = df.copy()
    df['study_efficiency'] = df['study_hours'] * df['class_attendance'] / 100
    df['sleep_study_ratio'] = df['study_hours'] / (df['sleep_hours'] + 0.001)  # avoid division by zero
    
    # Binning
    df['age_group'] = pd.cut(df['age'], bins=[16, 19, 22, 25], labels=['teen', 'young_adult', 'adult'])
    df['study_hours_category'] = pd.cut(df['study_hours'], bins=[0, 3, 6, 10], labels=['low', 'medium', 'high'])
    
    # Ordinal encoding for ordinal variables
    sleep_quality_map = {'poor': 0, 'average': 1, 'good': 2}
    facility_map = {'low': 0, 'medium': 1, 'high': 2}
    difficulty_map = {'easy': 0, 'moderate': 1, 'hard': 2}
    study_map = {'low': 0, 'medium': 1, 'high': 2}
    age_map = {'teen': 0, 'young_adult': 1, 'adult': 2}
    
    df['sleep_quality_encoded'] = df['sleep_quality'].map(sleep_quality_map)
    df['facility_rating_encoded'] = df['facility_rating'].map(facility_map)
    df['exam_difficulty_encoded'] = df['exam_difficulty'].map(difficulty_map)
    df['study_hours_category_encoded'] = df['study_hours_category'].map(study_map).astype('int64')
    df['age_group_encoded'] = df['age_group'].map(age_map).astype('int64')
    
    # Binary encoding
    df = pd.get_dummies(df, columns =['gender', 'course', 'study_method', 'internet_access'], drop_first=True)

    df.drop(columns=['sleep_quality', 'facility_rating', 'exam_difficulty', 'age_group', 'study_hours_category'], axis=1, inplace=True)
    '''
    # Target encoding (if you have target variable)
    if 'exam_score' in df.columns:
        course_mean_score = df.groupby('course')['exam_score'].mean().to_dict()
        df['course_mean_score'] = df['course'].map(course_mean_score)
        
        method_mean_score = df.groupby('study_method')['exam_score'].mean().to_dict()
        df['method_mean_score'] = df['study_method'].map(method_mean_score)
    '''
    # Interaction features
    df['attendance_sleep_interaction'] = df['class_attendance'] * df['sleep_quality_encoded']
    
    # Polynomial features
    df['study_hours_squared'] = df['study_hours'] ** 2
    df['attendance_squared'] = df['class_attendance'] ** 2
    
    return df

In [6]:
model = xgb.XGBRegressor(
    n_estimators=11859,
    max_depth=5,
    learning_rate=0.010120403942776919,
    subsample=0.8257919033762509,
    colsample_bytree=0.6463609420443598,
    min_child_weight=5,
    gamma=0.3227280726751745,
    reg_alpha=2.393775336268971,
    reg_lambda=3.556649603095873,
    random_state=42,  # Added for reproducibility
    n_jobs=-1,  # Use all CPU cores
)

In [7]:
train = pd.read_csv('/kaggle/input/playground-series-s6e1/train.csv').set_index('id')

In [8]:
y_train_dataset = train['exam_score']

In [9]:
train_dataset = engineer_features(train)

In [10]:
train_dataset = train_dataset.drop(columns=['exam_score'])

In [11]:
model.fit(
        train_dataset, y_train_dataset,     
        verbose=True
    )

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6463609420443598, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=0.3227280726751745, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.010120403942776919,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=5, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=11859, n_jobs=-1,
             num_parallel_tree=None, ...)

In [12]:
test = pd.read_csv('/kaggle/input/playground-series-s6e1/test.csv')

In [13]:
test

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,630000,24,other,ba,6.85,65.2,yes,5.2,poor,group study,high,easy
1,630001,18,male,diploma,6.61,45.0,no,9.3,poor,coaching,low,easy
2,630002,24,female,b.tech,6.60,98.5,yes,6.2,good,group study,medium,moderate
3,630003,24,male,diploma,3.03,66.3,yes,5.7,average,mixed,medium,moderate
4,630004,20,female,b.tech,2.03,42.4,yes,9.2,average,coaching,low,moderate
...,...,...,...,...,...,...,...,...,...,...,...,...
269995,899995,21,other,b.com,2.55,82.3,yes,8.4,average,mixed,medium,hard
269996,899996,17,female,b.com,0.49,46.4,yes,8.8,good,mixed,low,easy
269997,899997,22,male,bba,6.62,74.7,yes,5.5,good,coaching,high,easy
269998,899998,22,other,ba,4.08,51.8,yes,8.7,poor,online videos,high,moderate


In [14]:
test_dataset = engineer_features(test)

In [15]:
test_dataset = test_dataset.set_index('id')

In [16]:
test_dataset

,age,study_hours,class_attendance,sleep_hours,study_efficiency,sleep_study_ratio,sleep_quality_encoded,facility_rating_encoded,exam_difficulty_encoded,study_hours_category_encoded,...,course_bca,course_diploma,study_method_group study,study_method_mixed,study_method_online videos,study_method_self-study,internet_access_yes,attendance_sleep_interaction,study_hours_squared,attendance_squared
id,,,,,,,,,,,,,,,,,,,,,
630000,24,6.85,65.2,5.2,4.46620,1.317054,0,2,0,2,...,False,False,True,False,False,False,True,0.0,46.9225,4251.04
630001,18,6.61,45.0,9.3,2.97450,0.710676,0,0,0,2,...,False,True,False,False,False,False,False,0.0,43.6921,2025.00
630002,24,6.60,98.5,6.2,6.50100,1.064344,2,1,1,2,...,False,False,True,False,False,False,True,197.0,43.5600,9702.25
630003,24,3.03,66.3,5.7,2.00889,0.531486,1,1,1,1,...,False,True,False,True,False,False,True,66.3,9.1809,4395.69
630004,20,2.03,42.4,9.2,0.86072,0.220628,1,0,1,0,...,False,False,False,False,False,False,True,42.4,4.1209,1797.76
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899995,21,2.55,82.3,8.4,2.09865,0.303535,1,1,2,0,...,False,False,False,True,False,False,True,82.3,6.5025,6773.29
899996,17,0.49,46.4,8.8,0.22736,0.055675,2,0,0,0,...,False,False,False,True,False,False,True,92.8,0.2401,2152.96
899997,22,6.62,74.7,5.5,4.94514,1.203418,2,2,0,2,...,False,False,False,False,False,False,True,149.4,43.8244,5580.09


In [17]:
predictions = model.predict(test_dataset)

In [18]:
predictions

array([71.51299, 69.80554, 88.17637, ..., 91.0947 , 55.31855, 67.11726],
      dtype=float32)

In [19]:
submission = pd.DataFrame({'id': list(test_dataset.index), 'exam_score': predictions})

In [20]:
submission

,id,exam_score
0,630000,71.512993
1,630001,69.805542
2,630002,88.176369
3,630003,55.955246
4,630004,47.706165
...,...,...
269995,899995,60.937065
269996,899996,39.946091
269997,899997,91.094704
269998,899998,55.318550


In [21]:
submission.to_csv('submission.csv', index=False)